# 下载 3D lung CosMx

Pentimalli et al., Cell Systems 2025。一个肺癌标本，六个 CosMx 测量平面；保留官方 Seurat 对象、平面文件、配准信息和 SHG。

[公开 Zenodo v3](https://zenodo.org/records/15240431) · [作者代码](https://github.com/rajewsky-lab/3D_lung)

默认下载该记录全部五个包，约 8.35 GiB；解压和格式转换需要额外磁盘空间。该数据不是实测 Visium。

## 1. 环境

Python ≥3.11。下面安装到当前 notebook 内核；已有依赖可跳过此单元格。
下载与转换共用同目录的 `download_utils.py`、`h5ad_utils.py`，无需先运行其他 notebook。
官方文件保存在 `official/`，清单在 `metadata/`，H5AD 在 `h5ad/`。
支持 HTTP 断点续传；有官方校验值时核验校验值，否则验证传输长度。重复运行会检查并跳过已完成文件。

In [ ]:
%pip install -q "anndata>=0.11,<0.13" pandas scipy h5py pyarrow


## 2. 路径与参数

In [ ]:
from pathlib import Path
import sys
import pandas as pd

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data" / "download").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从 DeepSpatial 项目目录或 data/download 运行。")
sys.path.insert(0, str(PROJECT_ROOT / "data" / "download"))
from download_utils import download_records, save_json

DATA_DIR = PROJECT_ROOT / "data" / "3D-lung-CosMx"
DATA_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_H5AD = False  # 可选：改为 True，在官方文件之外增加 H5AD；不修改原文件。
ZENODO_RECORD = "15240431"
FILES = None  # None=官方五个包；可显式选择文件名列表
EXTRACT_ARCHIVES = True
print("数据目录：", DATA_DIR)


## 3. 获取并查看文件清单

In [ ]:
from download_utils import zenodo_records

records = zenodo_records(DATA_DIR, record=ZENODO_RECORD, filenames=FILES)
display(pd.DataFrame(records)[["path", "size", "checksum"]])
print(f"下载 {sum(r['size'] for r in records) / 1024**3:.2f} GiB")


## 4. 下载、MD5 校验与解压

ZIP 中的分析代码只作为资料保存，不执行。

In [ ]:
download_records(records, DATA_DIR, extract=EXTRACT_ARCHIVES)


## 5. 可选：Seurat → 按切片 H5AD

需要已安装 `Rscript`，以及 R 包 `SeuratObject`（支持 Seurat v5）、`Matrix`。本步骤不自动安装 R。
`cosmx.rds` 读取后按 section 导出 Spatial/counts；Python 生成 `h5ad/section_*.h5ad`。
保留官方 celltypes、生态位和其他元数据，空间使用微米坐标，另增加 `cell_type`、`z_coord_um` 便于读取。
该步骤读取较大的 Seurat 对象并暂存 Matrix Market 文件，需预留内存和磁盘。

In [ ]:
if EXPORT_H5AD:
    from h5ad_utils import export_cosmx
    h5ad_files = export_cosmx(DATA_DIR)
    print(f"导出 {len(h5ad_files)} 个切片 H5AD")
else:
    print("官方 RDS/ZIP 已保存；EXPORT_H5AD=True 启用可选导出。")
